# Feature Selection Pipeline
---
### Variance · Correlation · Lasso · Borda Aggregation · Bootstrap Stability

## Configuration

### Imports

In [1]:
from collections import defaultdict
import random
import warnings

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping, log_evaluation
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")


### Seeds & constants

In [2]:
# Seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths
DATA_PATH = "data/sen/train.csv"
OUTPUT_FEATURES = "data/sen/selected_features.txt"
OUTPUT_TRAIN_SEL = "data/sen/train_selected.csv"

# Parameters
TARGET = "agb"
N_FEATURES = 12 # max number of features to select
N_SPLITS = 5 # KFold splits
CORR_THRESHOLD = 0.90 # max allowed pairwise correlation
VAR_THRESHOLD = 0.01 # min variance to keep a feature
BORDA_COVERAGE = 0.80 # cumulative Borda score target
MAX_ITER = 50000 # max iterations
N_BOOTSTRAP = 50 # number of bootstrap resamples
STABILITY_THRESHOLD = 0.70 # feature must appear in >= 70% of runs to be 'stable'
STABILITY_STRATEGY = 'strict'

# Columns to exclude from feature candidates
DROP_COLS = [
    "inventory_date", "start_date", "end_date", "geometry",
    "agb", "cagb", "cagb_category", "HCS",
    "latitude", "longitude", "latitude_proj", "longitude_proj"
]

print("Configuration loaded.")


Configuration loaded.


## Data Loading

### Load training data

In [3]:
train_data = pd.read_csv(DATA_PATH)
print(f"Dataset : {train_data.shape[0]} observations x {train_data.shape[1]} features")

feature_cols = [c for c in train_data.columns if c not in DROP_COLS]
print(f"Feature candidates : {len(feature_cols)}")

X_dev = train_data[feature_cols].copy()
y_dev = train_data[TARGET].copy()


Dataset : 133 observations x 90 features
Feature candidates : 79


## Feature Filtering

### Variance threshold filter

In [4]:
# Remove near-zero variance features (uninformative for any model)
vt = VarianceThreshold(threshold=VAR_THRESHOLD)
X_dev_vt = vt.fit_transform(X_dev)
feature_cols_vt = np.array(feature_cols)[vt.get_support()].tolist()
print(f"VarianceThreshold filter : From {X_dev.shape[1]} to {len(feature_cols_vt)} features")


VarianceThreshold filter : From 79 to 26 features


### Correlation filter

In [5]:
# Remove one feature from each highly correlated pair
X_dev_vt_df = pd.DataFrame(X_dev_vt, columns=feature_cols_vt)

## Feature-target absolute correlations (Pearson)
target_corr = X_dev_vt_df.corrwith(pd.Series(np.array(y_dev), name='y')).abs()

## Pairwise feature correlation matrix (upper triangle only)
corr_matrix = X_dev_vt_df.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

## Keep whichever has higher absolute correlation with y and drop the other
to_drop = set()
for col in upper.columns:
    correlated_with = upper.index[upper[col] > CORR_THRESHOLD].tolist()
    for row in correlated_with:
        if target_corr.get(col, 0) >= target_corr.get(row, 0):
            to_drop.add(row)
        else:
            to_drop.add(col)

cols_to_keep = [c for c in feature_cols_vt if c not in to_drop]

## Build index mapping for array slicing
col_to_idx = {col: i for i, col in enumerate(feature_cols_vt)}
keep_idx = [col_to_idx[c] for c in cols_to_keep]
X_dev_clean = X_dev_vt[:, keep_idx]
y_dev_arr = np.array(y_dev)

print(f"Correlation filter : From {len(feature_cols_vt)} to {len(cols_to_keep)} features")
if to_drop:
    print(f"  Removed ({len(to_drop)}) : {sorted(to_drop)}")


Correlation filter : From 26 to 15 features
  Removed (11) : ['B11', 'B12', 'BAIM', 'DPDD', 'VDDPI', 'VH', 'VHVVD', 'VHVVP', 'VHVVR', 'VV', 'WI2015']


###  LassoCV — optimal alpha search

In [6]:
# Lasso
scaler_lasso = RobustScaler()
X_dev_scaled = scaler_lasso.fit_transform(X_dev_clean)

lasso_cv = LassoCV(cv=N_SPLITS, random_state=SEED, n_jobs=-1, max_iter=MAX_ITER)
lasso_cv.fit(X_dev_scaled, y_dev_arr)
best_alpha = lasso_cv.alpha_
print(f"\nLasso — best alpha : {best_alpha:.6f}")



Lasso — best alpha : 0.049872


## Importance Computation

### KFold importance function

In [7]:
def get_importances_kfold(model_name, build_model_fn, X, y,
                           feature_names, n_splits=N_SPLITS, needs_scale_arg=False):
    """
    Compute normalized, rank-based feature importances via KFold CV.
    Each fold contributes a reciprocal-rank score (1/rank) per feature,
    averaged across folds. This makes scores comparable across model types.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    importances = np.zeros(len(feature_names))

    for train_idx, val_idx in kf.split(X):
        X_tr,  y_tr  = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx],   y[val_idx]

        model = build_model_fn()

        # Lasso
        if needs_scale_arg:
            pipe = Pipeline([
                ("scaler", RobustScaler()),
                ("model",  model)
            ])
            pipe.fit(X_tr, y_tr)
            fold_imp = np.abs(pipe.named_steps["model"].coef_)

        # LightGBM
        elif model_name == "LightGBM":
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[
                    lgb_early_stopping(stopping_rounds=30, verbose=False),
                    log_evaluation(-1)
                ]
            )
            fold_imp = model.feature_importances_

        # CatBoost
        elif model_name == "CatBoost":
            model.fit(
                X_tr, y_tr,
                eval_set=(X_val, y_val),
                use_best_model=True,
                verbose=False
            )
            fold_imp = model.get_feature_importance()

        # GradientBoosting
        else:
            model.fit(X_tr, y_tr)
            fold_imp = model.feature_importances_

        # Normalize to [0, 1] within the fold
        fold_imp = fold_imp / (fold_imp.sum() + 1e-10)

        # Reciprocal-rank scoring : best feature gets 1/1, second gets 1/2, etc.
        ranked_idx = np.argsort(fold_imp)[::-1]
        for rank_based, feat_idx in enumerate(ranked_idx):
            importances[feat_idx] += 1.0 / (rank_based + 1)

    # Average across folds and return as sorted Series
    return pd.Series(
        importances / n_splits,
        index=feature_names
    ).sort_values(ascending=False)

print("get_importances_kfold defined.")

get_importances_kfold defined.


### Models configuration

In [8]:
models_config = {
    "Lasso": (
        lambda: Lasso(alpha=best_alpha, max_iter=MAX_ITER, random_state=SEED),
        True
    ),
    "LightGBM": (
        lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1),
        False
    ),
    "CatBoost": (
        lambda: CatBoostRegressor(random_seed=SEED, verbose=0),
        False
    ),
    "GradientBoosting": (
        lambda: GradientBoostingRegressor(random_state=SEED),
        False
    ),
}

print(f"{len(models_config)} models configured.")


4 models configured.


### Run importance computation

In [9]:
all_importances = {}

print(f"\n{'═'*60}")
print("IMPORTANCES COMPUTATION (5-Fold CV per model)")
print(f"{'═'*60}")

for name, (build_fn, needs_scale) in models_config.items():
    try:
        print(f"\n  {name} ...")
        imp = get_importances_kfold(
            model_name = name,
            build_model_fn = build_fn,
            X = X_dev_clean,
            y = y_dev_arr,
            feature_names = cols_to_keep,
            n_splits = N_SPLITS,
            needs_scale_arg = needs_scale
        )
        all_importances[name] = imp

        print(f"  {'Rank':<5} {'Feature':<25} {'Score':>10}")
        print("  " + "-" * 42)
        for rank, (feat, score) in enumerate(imp.head(N_FEATURES).items(), 1):
            print(f"  {rank:<5} {feat:<25} {score:>10.6f}")

    except Exception as e:
        print(f"  Ignored {name} from error : {e}")

print(f"\nImportances computed for {len(all_importances)}/{len(models_config)} models.")



════════════════════════════════════════════════════════════
IMPORTANCES COMPUTATION (5-Fold CV per model)
════════════════════════════════════════════════════════════

  Lasso ...
  Rank  Feature                        Score
  ------------------------------------------
  1     DpRVIVV                     1.000000
  2     VVVHD                       0.500000
  3     AWEIsh                      0.333333
  4     NBSIMS                      0.206667
  5     AWEInsh                     0.166667
  6     MIRBI                       0.163333
  7     MuWIR                       0.140404
  8     TTVI                        0.138016
  9     BAI                         0.121190
  10    ARI                         0.106032
  11    S2REP                       0.096515
  12    VVVHS                       0.093651

  LightGBM ...
  Rank  Feature                        Score
  ------------------------------------------
  1     VVVHS                       0.900000
  2     S2REP                       0

## Borda Aggregation & Final Selection

### Borda score aggregation

In [10]:
# Aggregate rankings across models using Borda count
borda_scores = defaultdict(float)

for name, imp in all_importances.items():
    for rank_0based, feat in enumerate(imp.index):
        borda_scores[feat] += (1.0 / (rank_0based + 1)) / len(cols_to_keep)

borda_series = pd.Series(borda_scores).sort_values(ascending=False)

print(f"Borda scores computed across {len(all_importances)} models.")


Borda scores computed across 4 models.


### Automatic N selection (80% coverage)

In [11]:
# Select the minimum number of features covering BORDA_COVERAGE of total score
cumulative_ratio = borda_series.cumsum() / borda_series.sum()
N_auto = int((cumulative_ratio < BORDA_COVERAGE).sum()) + 1
N_final = min(N_auto, N_FEATURES)

print(f"Auto selection  : {N_auto} features cover {BORDA_COVERAGE:.0%} of Borda score")
print(f"N_FEATURES cap  : {N_FEATURES}")
print(f"N_final applied : {N_final}")

top_features = borda_series.head(N_final).index.tolist()
print(f"\nSelected features ({N_final}) :")
for i, f in enumerate(top_features, 1):
    print(f"  {i:>2}. {f}")


Auto selection  : 9 features cover 80% of Borda score
N_FEATURES cap  : 12
N_final applied : 9

Selected features (9) :
   1. VVVHS
   2. S2REP
   3. DpRVIVV
   4. AWEIsh
   5. BAI
   6. MuWIR
   7. VVVHD
   8. MIRBI
   9. NBSIMS


### Comparative rank table

In [12]:
# Cross-model rank comparison
model_names = list(all_importances.keys())

rank_df = pd.DataFrame({
    name: all_importances[name].rank(ascending=True).astype(int)
    for name in model_names
})
rank_df["mean_rank"] = rank_df[model_names].mean(axis=1)
rank_df["borda_score"] = borda_series
rank_df = rank_df.sort_values("borda_score", ascending=False)

print(f"\n Ranks comparative table — Top {N_final}")
print(rank_df.head(N_final).to_string())



 Ranks comparative table — Top 9
         Lasso  LightGBM  CatBoost  GradientBoosting  mean_rank  borda_score
VVVHS        4        15        15                15      12.25     0.205556
S2REP        5        14        14                12      11.25     0.089394
DpRVIVV     15         3         1                 4       5.75     0.081795
AWEIsh      13         5        13                13      11.00     0.072727
BAI          7        13         9                14      10.75     0.072487
MuWIR        9        12        12                 8      10.25     0.051190
VVVHD       14         4         5                 3       6.50     0.050078
MIRBI       10        10        11                11      10.50     0.048889
NBSIMS      12         9         7                 9       9.25     0.043122


## Bootstrap Stability Analysis

In [13]:
# For each resample, we repeat the full Borda aggregation on a bootstrap
# subsample of the training set and record which features are selected.
# Selection frequency = proportion of runs in which a feature appears
# in the top-N_final features. A stable feature (freq >= STABILITY_THRESHOLD)
# is robust to small dataset perturbations.

rng = np.random.default_rng(SEED)
n_samples = X_dev_clean.shape[0]
feature_names_clean = cols_to_keep  # features after variance + correlation filter

selection_counts = defaultdict(int)

for b in range(N_BOOTSTRAP):
    # Bootstrap resample (with replacement, same size)
    idx = rng.integers(0, n_samples, size=n_samples)
    X_boot = X_dev_clean[idx]
    y_boot = y_dev_arr[idx]

    boot_importances = {}

    for name, (build_fn, needs_scale) in models_config.items():
        try:
            imp = get_importances_kfold(
                model_name = name,
                build_model_fn = build_fn,
                X = X_boot,
                y = y_boot,
                feature_names = feature_names_clean,
                n_splits = N_SPLITS,
                needs_scale_arg = needs_scale,
            )
            boot_importances[name] = imp
        except Exception:
            pass

    if not boot_importances:
        continue

    # Borda aggregation on this resample
    boot_borda = defaultdict(float)
    for name, imp in boot_importances.items():
        for rank_0, feat in enumerate(imp.index):
            boot_borda[feat] += (1.0 / (rank_0 + 1)) / len(feature_names_clean)

    boot_borda_series = pd.Series(boot_borda).sort_values(ascending=False)
    selected_boot = boot_borda_series.head(N_final).index.tolist()

    for feat in selected_boot:
        selection_counts[feat] += 1

    if (b + 1) % 10 == 0:
        print(f"Bootstrap {b + 1}/{N_BOOTSTRAP} done")

print(f"\nBootstrap complete ({N_BOOTSTRAP} resamples).")


Bootstrap 10/50 done
Bootstrap 20/50 done
Bootstrap 30/50 done
Bootstrap 40/50 done
Bootstrap 50/50 done

Bootstrap complete (50 resamples).


In [14]:
# Stability report
stability_series = (
    pd.Series(selection_counts)
    .reindex(borda_series.index, fill_value=0)  # include features never selected
    .sort_values(ascending=False)
)
stability_freq = stability_series / N_BOOTSTRAP

# Split into stable vs unstable relative to our final selection
stable_features   = [f for f in top_features if stability_freq.get(f, 0) >= STABILITY_THRESHOLD]
unstable_features = [f for f in top_features if stability_freq.get(f, 0) <  STABILITY_THRESHOLD]

print(f"{'═'*55}")
print(f"BOOTSTRAP STABILITY REPORT ({N_BOOTSTRAP} resamples)")
print(f"{'═'*55}")
print(f"\n{'Rank':<5} {'Feature':<28} {'Freq':>6}  {'Status'}")
print("-" * 55)
for rank, feat in enumerate(top_features, 1):
    freq  = stability_freq.get(feat, 0)
    label = 'stable' if freq >= STABILITY_THRESHOLD else 'unstable'
    print(f"{rank:<5} {feat:<28} {freq:>5.0%}  {label}")

print(f"\nSummary : {len(stable_features)}/{len(top_features)} stable features "
      f"(threshold = {STABILITY_THRESHOLD:.0%})")

if unstable_features:
    print("\nUnstable features (consider reviewing or removing):")
    for f in unstable_features:
        print(f"- {f}  (freq = {stability_freq.get(f, 0):.0%})")


═══════════════════════════════════════════════════════
BOOTSTRAP STABILITY REPORT (50 resamples)
═══════════════════════════════════════════════════════

Rank  Feature                        Freq  Status
-------------------------------------------------------
1     VVVHS                         100%  stable
2     S2REP                          72%  stable
3     DpRVIVV                        96%  stable
4     AWEIsh                         94%  stable
5     BAI                            94%  stable
6     MuWIR                          64%  unstable
7     VVVHD                          96%  stable
8     MIRBI                          64%  unstable
9     NBSIMS                         42%  unstable

Summary : 6/9 stable features (threshold = 70%)

Unstable features (consider reviewing or removing):
- MuWIR  (freq = 64%)
- MIRBI  (freq = 64%)
- NBSIMS  (freq = 42%)


## Stability-aware final selection

In [15]:
# Apply stability strategy

if STABILITY_STRATEGY == 'strict':
    # Keep only features from top_features that meet the stability threshold
    top_features_final = [
        f for f in top_features
        if stability_freq.get(f, 0) >= STABILITY_THRESHOLD
    ]
    strategy_note = (
        f"Strict filter : From {len(top_features)} to {len(top_features_final)} features "
        f"(threshold = {STABILITY_THRESHOLD:.0%})"
    )

elif STABILITY_STRATEGY == 'hybrid':
    # Reweight Borda scores by stability frequency, then re-select top-N_final
    # stability_freq may not cover all features in borda_series (never selected
    # in any bootstrap run, which acts as a natural penalty)
    hybrid_scores = borda_series * stability_freq.reindex(borda_series.index, fill_value=0)
    hybrid_scores = hybrid_scores.sort_values(ascending=False)
    top_features_final = hybrid_scores.head(N_final).index.tolist()
    strategy_note = (
        f"Hybrid reweighting : Borda x stability_freq, top-{N_final} re-selected"
    )

else:
    raise ValueError(f"Unknown STABILITY_STRATEGY: {STABILITY_STRATEGY!r}. "
                     "Choose 'strict' or 'hybrid'.")

# Borda-only vs stability-aware
added = [f for f in top_features_final if f not in top_features]
removed = [f for f in top_features     if f not in top_features_final]

print(f"\n{strategy_note}")
print(f"\n{'Feature':<28} {'Borda':>8}  {'Stability':>10}  {'In final'}")
print("-" * 65)
all_candidates = list(dict.fromkeys(top_features + top_features_final))
for feat in all_candidates:
    b_score = borda_series.get(feat, 0)
    s_freq  = stability_freq.get(feat, 0)
    in_final = 'kept' if feat in top_features_final else 'removed'
    new_tag  = 'added' if feat in added else ''
    print(f"{feat:<28} {b_score:>8.5f}  {s_freq:>9.0%}  {in_final}{new_tag}")

if removed:
    print(f"\nRemoved : {removed}")
if added:
    print(f"Added : {added}")
if not removed and not added:
    print("\nNo change between Borda-only and stability-aware selection.")

top_features = top_features_final
print(f"\nTop features ({len(top_features)} features) : {top_features}")



Strict filter : From 9 to 6 features (threshold = 70%)

Feature                         Borda   Stability  In final
-----------------------------------------------------------------
VVVHS                         0.20556       100%  kept
S2REP                         0.08939        72%  kept
DpRVIVV                       0.08179        96%  kept
AWEIsh                        0.07273        94%  kept
BAI                           0.07249        94%  kept
MuWIR                         0.05119        64%  removed
VVVHD                         0.05008        96%  kept
MIRBI                         0.04889        64%  removed
NBSIMS                        0.04312        42%  removed

Removed : ['MuWIR', 'MIRBI', 'NBSIMS']

Top features (6 features) : ['VVVHS', 'S2REP', 'DpRVIVV', 'AWEIsh', 'BAI', 'VVVHD']


## Save selected features & filtered dataset

In [16]:
# Save feature list for downstream notebooks
with open(OUTPUT_FEATURES, "w", encoding="utf-8") as f:
    for feat in top_features:
        f.write(feat + "\n")

print(f"Selected features saved : {OUTPUT_FEATURES}")

# Save filtered train set (selected features + target)
train_selected = train_data[top_features + [TARGET]].copy()
train_selected.to_csv(OUTPUT_TRAIN_SEL, index=False)
print(f"Filtered train set saved : {OUTPUT_TRAIN_SEL} — shape {train_selected.shape}")

print("\nFinal selection summary :")
print(f"  Input features : {len(feature_cols)}")
print(f"  After variance : {len(feature_cols_vt)}")
print(f"  After correlation : {len(cols_to_keep)}")
print(f"  After borda : {N_final}")
print(f"  Final selected : {len(top_features)}")
print(f"\n  Features : {top_features}")


Selected features saved : data/sen/selected_features.txt
Filtered train set saved : data/sen/train_selected.csv — shape (133, 7)

Final selection summary :
  Input features : 79
  After variance : 26
  After correlation : 15
  After borda : 9
  Final selected : 6

  Features : ['VVVHS', 'S2REP', 'DpRVIVV', 'AWEIsh', 'BAI', 'VVVHD']
